# Palm Oil — Data Exploration

A visual tour of the palm oil price series behind the PKO experiment, aimed at two
decisions: **which series to forecast**, and **which 7 cutoffs to forecast from**.

The target is FRED `PPOILUSDM` — the IMF global benchmark palm oil price, monthly,
USD per metric ton, registered with true publication dates by `pko.data`.

See [`DATA.md`](DATA.md) for the full survey of what FRED carries and why this series
was chosen. Every chart below is interactive: **click a legend entry to hide a series**,
drag to zoom, double-click to reset.


---
## 1. Load the price series


In [ ]:
from __future__ import annotations

import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env")

from pko.data import PALM_OIL_SERIES_ID, build_palm_oil_service
from pko.plots import (
    DEFAULT_CUTOFFS,
    plot_cutoff_windows,
    plot_information_gap,
    plot_monthly_changes,
    plot_oil_complex,
    plot_price_history,
)


svc = build_palm_oil_service(cache_dir=ROOT / "data" / "fred")
as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)
prices = svc.get_series(PALM_OIL_SERIES_ID, as_of=as_of)

print(f"{len(prices)} monthly observations")
print(f"span   : {prices.timestamp.min():%Y-%m} -> {prices.timestamp.max():%Y-%m}")
print(f"price  : ${prices.value.min():.0f} to ${prices.value.max():.0f} per tonne")
prices.tail()

414 monthly observations
span   : 1992-01 -> 2026-06
price  : $185 to $1653 per tonne


,timestamp,value,released_at
409,2026-02-01,1033.768394,2026-03-24
410,2026-03-01,1121.177897,2026-04-15
411,2026-04-01,1137.410862,2026-06-05
412,2026-05-01,1130.026757,2026-06-05
413,2026-06-01,1108.681096,2026-07-13


The `released_at` column is the point of this whole setup — it is the date FRED
*published* each price, not the month the price refers to. June 2026's price was
published on 2026-07-13, six weeks after the timestamp says.

That column is what stops the harness handing a model a price that did not exist yet.


---
## 2. The signal itself

Full history since 2015, with the 7 candidate cutoffs marked and the two publication
blackouts shaded in red. Drag the range slider at the bottom to zoom into any period.


In [ ]:
plot_price_history(prices, cutoffs=DEFAULT_CUTOFFS)

Things to look for:

- The **2021–2022 spike** to \$1,653 and the crash back to \$903 — the Indonesian export ban.
- The **red bands** are periods when FRED published nothing at all. Note that the first
  one covers the entire export-ban episode.
- The recent climb through 2026 to around \$1,100.


---
## 3. Month-over-month change

The same series as returns, which is what a forecaster is really trying to predict.
Blue is up, red is down.


In [ ]:
plot_monthly_changes(prices, start="2020-01-01")

July 2022 is **−31.7%** — the largest single month in the data. It sits inside a
blackout, so nobody could see it happening at the time.


---
## 4. Do the candidate cutoffs actually look right?

Each shaded band is one cutoff's 6-month forecast window. Orange bands are the
"event" cutoffs, blue are "quiet".

**This chart is the check on the cutoff choice.** An event window should visibly
contain a shock; a quiet window should look flat. If one doesn't, swap it.


In [ ]:
plot_cutoff_windows(prices)

In [ ]:
# The cutoffs and why each was picked — edit this list to try alternatives.
pd.DataFrame([{"cutoff": c.date[:7], "kind": c.kind, "reason": c.label} for c in DEFAULT_CUTOFFS])

,cutoff,kind,reason
0,2021-05,event,"June 2021 crash, -16.6%"
1,2022-01,event,"Indonesia export ban, -29.4% over 6mo"
2,2023-04,event,"May 2023 correction, -10.7%"
3,2024-09,event,"Oct 2024 rally, +9.7%"
4,2023-07,quiet,"calmest window, max move 4.1%"
5,2024-11,quiet,max move 8.7%
6,2025-08,quiet,max move 5.9%


---
## 5. What the model can actually see

This is the chart that makes the publication lag concrete.

The dotted grey line is what really happened. Each coloured line is the history that
was **published** as of one cutoff — where it stops is the newest price a forecaster
had on that date.

The horizontal distance between where a coloured line ends and its cutoff is the
information gap. Normally 2 months; far worse inside a blackout.


In [ ]:
plot_information_gap(svc, PALM_OIL_SERIES_ID)

In [ ]:
# The gap at every candidate cutoff, as a table.
rows = []
for c in DEFAULT_CUTOFFS:
    seen = svc.get_series(PALM_OIL_SERIES_ID, as_of=c.timestamp.to_pydatetime())
    last = seen.timestamp.max()
    gap = (c.timestamp.year - last.year) * 12 + (c.timestamp.month - last.month)
    rows.append(
        {
            "cutoff": c.date[:7],
            "kind": c.kind,
            "newest price": f"{last:%Y-%m}",
            "gap (months)": gap,
            "h=1 really means": f"{gap + 1} months past last data",
        }
    )
pd.DataFrame(rows)

,cutoff,kind,newest price,gap (months),h=1 really means
0,2021-05,event,2021-03,2,3 months past last data
1,2022-01,event,2021-11,2,3 months past last data
2,2023-04,event,2023-02,2,3 months past last data
3,2024-09,event,2024-07,2,3 months past last data
4,2023-07,quiet,2023-05,2,3 months past last data
5,2024-11,quiet,2024-09,2,3 months past last data
6,2025-08,quiet,2025-06,2,3 months past last data


A nominal horizon of 1 is really a **3-month** extrapolation once the 2-month gap is
counted. Horizons 1–6 therefore span 3 to 8 months of real forecast distance — worth
stating explicitly in any writeup, or `h=1` reads as an easy nowcast when it isn't.


---
## 6. The rest of the edible-oil complex

Palm oil against the three other IMF oils. Same units, same release calendar, same
leak-safe handling — so they are cheap covariates if they carry signal.

Click legend entries to isolate pairs and judge whether they move together.


In [ ]:
from aieng.forecasting.data.adapters import FREDAdapter


FRED_CACHE = ROOT / "data" / "fred"
COMPLEX = {"Soybean oil": "PSOILUSDM", "Sunflower oil": "PSUNOUSDM", "Rapeseed oil": "PROILUSDM"}

oils = {"Palm oil": prices}
for name, fred_id in COMPLEX.items():
    oils[name] = FREDAdapter(fred_id, cache_dir=FRED_CACHE).fetch()

plot_oil_complex(oils)

In [ ]:
# Correlation of monthly returns — does the complex actually co-move?
returns = pd.DataFrame(
    {name: frame.set_index("timestamp")["value"].pct_change() for name, frame in oils.items()}
).dropna()
returns.loc["2015-01-01":].corr().round(2)

,Palm oil,Soybean oil,Sunflower oil,Rapeseed oil
Palm oil,1.00,0.54,0.60,0.44
Soybean oil,0.54,1.00,0.55,0.55
Sunflower oil,0.60,0.55,1.00,0.62
Rapeseed oil,0.44,0.55,0.62,1.00


---
## 7. Where this leaves us

| Decision | Status |
|---|---|
| Target series | `PPOILUSDM` — palm oil, monthly, 414 observations |
| Leak safety | True publication dates attached; verified in `pko.data` |
| Horizons | 1–6 months, which is 3–8 months past the last known price |
| Cutoffs | 7 candidates in `DEFAULT_CUTOFFS` — check them in section 4 |

**Next:** a naive `LastValuePredictor` baseline over these cutoffs, to establish the
floor every other model has to beat.

**Still open:** FRED has no palm *kernel* oil series, so this use case forecasts palm
oil. See [`DATA.md`](DATA.md).
